# Update log
2024/08/29
- Test without Responsivity and/or baseline
- Test with reduced number of features
- Baseline alone gives up to 75% accuracy
- Suspect data distribution due to always running experiment in 0,1,2,3,4
- Run experiment in De Bruijn sequence to balance the adjacent experiment channels
---

In [1]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import json

spk_data = "D:\\code\\uom_explore\\model_input\\3_features.csv"
spk_pca_data = "D:\\code\\uom_explore\\model_input\\pca_df.csv"
spk_20 = "D:\\code\\uom_explore\\data_science\\reduced\\20_140_195_df.csv"
spk_full = "D:\\code\\uom_explore\\data_science\\df.csv"
# debruijn_1 = "D:\\code\\uom_explore\\processed_data\\metrics_exp_brujin_seq_1.csv"
debruijn_1 = "D:\\code\\uom_explore\\processed_data\\metrics_de_brujin_large_even_1.csv"

hkr_wsl_data = "/home/hk-wsl/code/uom_explore/model_input/feature_matrix.csv"
hkr_pca_data = "/home/hk-wsl/code/uom_explore/model_input/feature_pca.csv"

spk_json = "/home/gavinlouuu/coding/uom_explore/data_science/parameter.json"
hkr_wsl_json = "/home/hk-wsl/code/uom_explore/data_science/parameter.json"

data_path = debruijn_1
param_path = spk_json

with open('parameter.json','r') as file:
    params = json.load(file)

# Hyperparameters
# Extract parameters from the JSON object
hidden_size = params['mlp']['hidden_size']
ground_truth = params['ground_truth']
num_epochs = params['mlp']['num_epochs']
batch_size = params['mlp']['batch_size']
learning_rate = params['mlp']['learning_rate']
momentum_value = params['mlp']['momentum_value']
dropout_rate = params['mlp']['dropout']
weight_decay = params['mlp']['weight_decay']

scheduler_params = params['mlp']['scheduler']

df = pd.read_csv(data_path)
# drop experiment_id column
# df.drop('experiment_id', axis=1, inplace=True)
print(type(df))
df.head()



<class 'pandas.core.frame.DataFrame'>


,experiment_id,channel_id,baseline_140,baseline_143,baseline_146,baseline_149,baseline_152,baseline_155,baseline_158,baseline_161,...,temperature_max,temperature_std,humidity_mean,humidity_min,humidity_max,humidity_std,pressure_mean,pressure_min,pressure_max,pressure_std
0,20240903124659s1c0r0,0,1304.321762,851.782734,822.821395,861.828490,915.179956,974.803680,1036.697018,1097.286209,...,30.46,0.357056,46.111294,44.70,47.49,0.858415,100439.741176,100433.0,100443.0,2.018680
1,20240903124734s1c1r0,1,1089.128873,686.398724,664.426060,698.958067,745.947882,798.561828,855.173888,921.501706,...,30.63,0.059947,46.180423,43.62,48.37,1.749974,100440.169014,100438.0,100442.0,0.925603
2,20240903124810s1c2r0,2,933.216271,584.428862,567.101092,598.706476,641.556892,688.345655,740.002441,805.943269,...,30.78,0.049209,43.093488,42.50,44.33,0.573540,100439.825581,100439.0,100441.0,0.617240
3,20240903124846s1c3r0,3,862.767043,544.731229,530.892613,561.774378,603.985331,649.670223,701.950674,766.896687,...,30.87,0.040033,44.220000,42.44,45.64,1.130297,100439.452055,100438.0,100442.0,0.866684
4,20240903124921s1c4r0,4,1193.223012,759.743885,752.950952,805.589437,869.654724,943.912449,1026.417458,1131.115262,...,30.97,0.039643,46.049886,43.03,49.10,1.974855,100437.340909,100435.0,100440.0,1.240007


## Load all features

In [2]:
# Get all column names from the DataFrame
all_columns = df.columns.tolist()

# Remove 'channel_id' and the ground truth from the list of features
features = [col for col in all_columns if col != 'experiment_id' and col != ground_truth]

# X includes all features
X = df[features]

# Print the features
print("Features:")
print(json.dumps(features, indent=2))



Features:
[
  "baseline_140",
  "baseline_143",
  "baseline_146",
  "baseline_149",
  "baseline_152",
  "baseline_155",
  "baseline_158",
  "baseline_161",
  "baseline_164",
  "baseline_167",
  "baseline_170",
  "baseline_173",
  "baseline_176",
  "baseline_179",
  "baseline_182",
  "baseline_185",
  "baseline_188",
  "baseline_191",
  "baseline_194",
  "baseline_197",
  "baseline_200",
  "baseline_203",
  "baseline_206",
  "baseline_209",
  "baseline_212",
  "baseline_215",
  "baseline_218",
  "baseline_221",
  "baseline_224",
  "baseline_227",
  "baseline_230",
  "baseline_233",
  "baseline_236",
  "baseline_239",
  "baseline_242",
  "baseline_245",
  "baseline_248",
  "baseline_250",
  "max_reaction_R_140",
  "max_reaction_R_143",
  "max_reaction_R_146",
  "max_reaction_R_149",
  "max_reaction_R_152",
  "max_reaction_R_155",
  "max_reaction_R_158",
  "max_reaction_R_161",
  "max_reaction_R_164",
  "max_reaction_R_167",
  "max_reaction_R_170",
  "max_reaction_R_173",
  "max_reaction_

# Adjust features

## Select settings to keep

In [3]:
# # Preserve BME features 
# bme_features = [col for col in features if any(suffix in col for suffix in ['_min','_max','_mean','_std'])]

# # Extract all unique numbers from feature names
# feature_numbers = set()
# for feature in features:
#     parts = feature.split('_')
#     if len(parts) > 1 and parts[-1].isdigit():
#         feature_numbers.add(int(parts[-1]))

# # Sort the numbers
# sorted_numbers = sorted(feature_numbers)

# # Select settings to keep
# settings_to_keep = []

# # Filter the features to keep only the selected settings
# features_to_keep = [feature for feature in features if feature.split('_')[-1].isdigit() and int(feature.split('_')[-1]) in settings_to_keep]

# # Add back the BME features
# features_to_keep.extend(bme_features)

# # Update the features list
# features = features_to_keep

# # Print the features
# print("Features:")
# print(json.dumps(features, indent=2))

## Select features to remove

In [4]:
# Remove features with _min, _max, and _std suffixes
features_to_keep = [col for col in features if not any(suffix in col for suffix in ['_std'])]

# Update the features list
features = features_to_keep

# Update X dataframe to only include the kept features
X = df[features]

# Update input_size
input_size = len(features)

# print(f"Features after removing '_min', '_max' and '_std' suffixes:")
print(json.dumps(features, indent=2))
# print(f"New input size: {input_size}")



[
  "baseline_140",
  "baseline_143",
  "baseline_146",
  "baseline_149",
  "baseline_152",
  "baseline_155",
  "baseline_158",
  "baseline_161",
  "baseline_164",
  "baseline_167",
  "baseline_170",
  "baseline_173",
  "baseline_176",
  "baseline_179",
  "baseline_182",
  "baseline_185",
  "baseline_188",
  "baseline_191",
  "baseline_194",
  "baseline_197",
  "baseline_200",
  "baseline_203",
  "baseline_206",
  "baseline_209",
  "baseline_212",
  "baseline_215",
  "baseline_218",
  "baseline_221",
  "baseline_224",
  "baseline_227",
  "baseline_230",
  "baseline_233",
  "baseline_236",
  "baseline_239",
  "baseline_242",
  "baseline_245",
  "baseline_248",
  "baseline_250",
  "max_reaction_R_140",
  "max_reaction_R_143",
  "max_reaction_R_146",
  "max_reaction_R_149",
  "max_reaction_R_152",
  "max_reaction_R_155",
  "max_reaction_R_158",
  "max_reaction_R_161",
  "max_reaction_R_164",
  "max_reaction_R_167",
  "max_reaction_R_170",
  "max_reaction_R_173",
  "max_reaction_R_176",
  

# Data split and scale

In [5]:
# Preview features
X = X[features]
# print(X.head())


In [6]:
input_size = len(X.columns)  # removing the ground truth from the number of columns counted
num_classes = df[ground_truth].nunique()
print(f"Number of classes: {num_classes}")
print(f"Number of features: {input_size}")

# Preview ground truth
y = df[ground_truth]


# Split into training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)  # This makes 60%, 20%, 20%

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler to the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to validation and test sets
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert arrays to NumPy arrays (no need to convert to PyTorch tensors)
X_train_np = X_train_scaled
y_train_np = y_train.to_numpy()
X_val_np = X_val_scaled
y_val_np = y_val.to_numpy()
X_test_np = X_test_scaled
y_test_np = y_test.to_numpy()


Number of classes: 5
Number of features: 123


# Random Forest

In [7]:
n_estimators = params['rf']['n_estimators']
random_state = params['rf']['random_state']
# Initialize the model
rf_model = RandomForestClassifier(
    n_estimators=n_estimators,
    random_state=random_state
)


# Function to predict using the Random Forest model
def predict_rf(model, data):
    return model.predict(data)



# Function to train and evaluate the Random Forest model
def train_and_evaluate_rf(model, X_train, y_train, X_val, y_val):
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions on training and validation sets
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    # Calculate accuracies
    train_accuracy = accuracy_score(y_train, y_train_pred)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    
    print(f'Train Accuracy: {train_accuracy:.4f}, Validation Accuracy: {val_accuracy:.4f}')
    
    return model, train_accuracy, val_accuracy

# Train and evaluate the Random Forest model
rf_model, rf_train_accuracy, rf_val_accuracy = train_and_evaluate_rf(
    rf_model, 
    X_train_scaled, y_train, 
    X_val_scaled, y_val
)

# Evaluate on the test set
y_test_pred = predict_rf(rf_model, X_test_scaled)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')


Train Accuracy: 1.0000, Validation Accuracy: 0.7396
Test Accuracy: 0.6667


# Gradient Boost

In [8]:
# Import necessary libraries
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Define parameters for Gradient Boosting
n_estimators = params['gb']['n_estimators']
learning_rate = params['gb']['learning_rate']
max_depth = params['gb']['max_depth']
random_state = params['gb']['random_state']

# Initialize the Gradient Boosting model
gb_model = GradientBoostingClassifier(
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    max_depth=max_depth,
    random_state=random_state
)

# Function to predict using the Gradient Boosting model
def predict_gb(model, data):
    return model.predict(data)

# Function to train and evaluate the Gradient Boosting model
def train_and_evaluate_gb(model, X_train, y_train, X_val, y_val):
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions on training and validation sets
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    # Calculate accuracies
    train_accuracy = accuracy_score(y_train, y_train_pred)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    
    print(f'Train Accuracy: {train_accuracy:.4f}, Validation Accuracy: {val_accuracy:.4f}')
    
    return model, train_accuracy, val_accuracy

# Train and evaluate the Gradient Boosting model
gb_model, gb_train_accuracy, gb_val_accuracy = train_and_evaluate_gb(
    gb_model, 
    X_train_scaled, y_train, 
    X_val_scaled, y_val
)

# Evaluate on the test set
y_test_pred = predict_gb(gb_model, X_test_scaled)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')

Train Accuracy: 1.0000, Validation Accuracy: 0.6979
Test Accuracy: 0.6771


# Stacking Ensemble

In [9]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

# Split the data into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Define the base models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

# Define the meta-model with increased max_iter and scaling
meta_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

# Create the stacking ensemble
stacking_ensemble = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)

# Train the stacking ensemble
stacking_ensemble.fit(X_train, y_train)

# Evaluate the model on the validation set
y_val_pred = stacking_ensemble.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f'Validation Accuracy: {val_accuracy:.4f}')

# Cross-validation scores
cv_scores = cross_val_score(stacking_ensemble, X_train, y_train, cv=5)
print(f'Cross-Validation Scores: {cv_scores}')
print(f'Mean CV Score: {np.mean(cv_scores):.4f}')

# Confusion Matrix for validation set
conf_matrix_val = confusion_matrix(y_val, y_val_pred)
print('Confusion Matrix (Validation):')
print(conf_matrix_val)

# Classification Report for validation set
class_report_val = classification_report(y_val, y_val_pred)
print('Classification Report (Validation):')
print(class_report_val)

# Evaluate the model on the test set
y_test_pred = stacking_ensemble.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')

# Confusion Matrix for test set
conf_matrix_test = confusion_matrix(y_test, y_test_pred)
print('Confusion Matrix (Test):')
print(conf_matrix_test)

# Classification Report for test set
class_report_test = classification_report(y_test, y_test_pred)
print('Classification Report (Test):')
print(class_report_test)

# Feature Importances from base models
for name, model in base_models:
    if hasattr(model, 'feature_importances_'):
        print(f'Feature importances for {name}:')
        print(model.feature_importances_)

Validation Accuracy: 0.7083
Cross-Validation Scores: [0.72727273 0.81818182 0.79220779 0.77631579 0.75      ]
Mean CV Score: 0.7728
Confusion Matrix (Validation):
[[ 7 10  0  2  1]
 [ 4 10  0  0  0]
 [ 0  0 11  0  1]
 [ 0  0  1  9  0]
 [ 1  1  0  0 14]]
Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.58      0.35      0.44        20
           1       0.48      0.71      0.57        14
           2       0.92      0.92      0.92        12
           3       0.82      0.90      0.86        10
           4       0.88      0.88      0.88        16

    accuracy                           0.71        72
   macro avg       0.73      0.75      0.73        72
weighted avg       0.72      0.71      0.70        72

Test Accuracy: 0.6250
Confusion Matrix (Test):
[[2 3 0 0 0]
 [4 0 0 0 0]
 [0 0 6 1 0]
 [0 0 1 0 0]
 [0 0 0 0 7]]
Classification Report (Test):
              precision    recall  f1-score   support

           0       0.33

# Voting Classifier

In [10]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

# First, split the data into train+val and test sets
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Then split the train+val set into separate train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)  # 0.25 x 0.8 = 0.2

# Define the base models with scaling for Logistic Regression
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('lr', make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))
]

# Create the voting ensemble
voting_ensemble = VotingClassifier(estimators=base_models, voting='hard')  # 'soft' for probability averaging

# Train the voting ensemble
voting_ensemble.fit(X_train, y_train)

# Evaluate on validation set
y_val_pred = voting_ensemble.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f'Validation Accuracy: {val_accuracy:.4f}')

# Cross-validation scores
cv_scores = cross_val_score(voting_ensemble, X_train, y_train, cv=5)
print(f'Cross-Validation Scores: {cv_scores}')
print(f'Mean CV Score: {np.mean(cv_scores):.4f}')

# Confusion Matrix for validation set
conf_matrix_val = confusion_matrix(y_val, y_val_pred)
print('Validation Confusion Matrix:')
print(conf_matrix_val)

# Classification Report for validation set
class_report_val = classification_report(y_val, y_val_pred)
print('Validation Classification Report:')
print(class_report_val)

# Final evaluation on test set
y_test_pred = voting_ensemble.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'\nTest Accuracy: {test_accuracy:.4f}')

# Confusion Matrix for test set
conf_matrix_test = confusion_matrix(y_test, y_test_pred)
print('Test Confusion Matrix:')
print(conf_matrix_test)

# Classification Report for test set
class_report_test = classification_report(y_test, y_test_pred)
print('Test Classification Report:')
print(class_report_test)

Validation Accuracy: 0.7396
Cross-Validation Scores: [0.86206897 0.75862069 0.70175439 0.70175439 0.75438596]
Mean CV Score: 0.7557
Validation Confusion Matrix:
[[ 9 10  0  0  0]
 [ 7 10  0  3  0]
 [ 0  0 18  0  0]
 [ 3  1  1 18  0]
 [ 0  0  0  0 16]]
Validation Classification Report:
              precision    recall  f1-score   support

           0       0.47      0.47      0.47        19
           1       0.48      0.50      0.49        20
           2       0.95      1.00      0.97        18
           3       0.86      0.78      0.82        23
           4       1.00      1.00      1.00        16

    accuracy                           0.74        96
   macro avg       0.75      0.75      0.75        96
weighted avg       0.74      0.74      0.74        96


Test Accuracy: 0.6875
Test Confusion Matrix:
[[ 9 12  0  3  1]
 [ 7 11  0  0  0]
 [ 0  0 17  1  1]
 [ 1  1  1  8  0]
 [ 2  0  0  0 21]]
Test Classification Report:
              precision    recall  f1-score   support

     